# COMP5339 Assignment 1 — Data Augmentation (Task 3) — Owner D

**Purpose:** enrich NSW **DC (fast) chargers** with one or more attributes absent from the
Transport for NSW dataset — plug types, pricing, operator detail, or number of bays — by querying
an external source programmatically.

**Target: at least 50% of DC charger locations.** For this dataset that is **216 of 432**.

Reads `data/interim/ev_chargers_cleaned.csv` (from `integration_quality_B.ipynb`), caches external
responses to `data/external/`, and writes matched output to `data/interim/`. It does not modify its
input.

## Running this notebook

Prerequisites, in order:

1. `data_acquisition.ipynb` has populated `data/raw/`.
2. `integration_quality_B.ipynb` has written `data/interim/ev_chargers_cleaned.csv`.
3. **An API key exists in `.env`.** There is no `.env` in the repo yet — copy `.env.example`,
   register with your chosen source, and fill it in. Nothing below runs without this.

Paths are resolved by searching upward for the repository root, so the notebook works regardless of
where Jupyter was launched. Do not reintroduce `Path.cwd()`: Jupyter starts in `notebooks/`, and
that assumption previously wrote 69 MB into the wrong directory.

In [2]:
from pathlib import Path
import json
import math
import os
import re

import numpy as np
import pandas as pd


def _repo_root() -> Path:
    """Locate the repository root, wherever Jupyter was launched from."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(
        f"Could not find the repository root (no pyproject.toml above {here})."
    )


PROJECT_ROOT = _repo_root()
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CSV = INTERIM_DIR / "ev_chargers_cleaned.csv"
DC_SUBSET_CSV = INTERIM_DIR / "dc_chargers.csv"
AUGMENTED_CSV = INTERIM_DIR / "dc_chargers_augmented.csv"
SUMMARY_JSON = INTERIM_DIR / "augmentation_summary.json"

# Cache of raw external responses. Unlike the other data/ directories this one
# IS committed, so the rest of the group can work without their own API key.
OCM_CACHE = EXTERNAL_DIR / "external_charger_details.csv"
OSM_CACHE = EXTERNAL_DIR / "osm_charging_stations.csv"
EXTERNAL_CACHE = OCM_CACHE  # kept: earlier cells refer to this name

def load_env(path: Path) -> None:
    """Minimal .env loader -- avoids a python-dotenv dependency.

    os.environ does not read .env by itself, so this must run before any cell
    that reads a key. Existing environment variables win over the file.
    """
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#") or "=" not in stripped:
            continue
        name, _, value = stripped.partition("=")
        os.environ.setdefault(name.strip(), value.strip())


load_env(PROJECT_ROOT / ".env")

assert CLEANED_CSV.exists(), f"Run integration_quality_B.ipynb first; {CLEANED_CSV} is missing."
print(f"Project root: {PROJECT_ROOT}")
print(f"Reading:      {CLEANED_CSV}")

Project root: D:\Data Engineering\comp5339-a1
Reading:      D:\Data Engineering\comp5339-a1\data\interim\ev_chargers_cleaned.csv


## 1. Derive the DC charger subset

The ≥50% target is measured against DC chargers, so this subset defines the denominator for
everything below. It is written to `data/interim/dc_chargers.csv` rather than kept as a local
variable, so that B's quality report and this notebook count the same population.

Expected counts in the December 2025 release, for assertion:

| | |
|---|---|
| all rows | 1,958 |
| `Charger_Type` | AC 1,427 · **DC 433** · Upcoming 98 |
| `duplicate_status` | active 1,950 · flagged_manual_review 6 · superseded 2 |
| **DC excluding superseded** | **432** → target **216** |

Two decisions to make explicitly and record for the report:

- **`superseded` rows — exclude.** These are resolved duplicates; the retained record already
  represents the site, and augmenting both would double-count.
- **The 6 `flagged_manual_review` rows — your call.** They are awaiting human validation. Including
  them inflates the denominator and risks matching a record that may later be withdrawn; excluding
  them narrows the population you are assessed on. Either is defensible; state which and why.

In [3]:
# Load the cleaned data and select the DC subset that defines the >=50% target.

EXPECTED_INPUT_ROWS = 1958  # December 2025 release, per integration_quality_B.ipynb

# TODO(D): confirm this decision and record the reasoning -- the report needs it.
# The two flagged_manual_review DC rows share one address (University of Wollongong,
# Northfields Avenue) but carry different operators, so they are probably one physical
# charger B could not resolve. Keeping both may double-count that site, and would mean
# augmenting it twice. Worth checking with B before settling.
#   True  -> 432 chargers, target 216   (current behaviour)
#   False -> 430 chargers, target 215
INCLUDE_FLAGGED_FOR_REVIEW = True

raw = pd.read_csv(CLEANED_CSV, dtype=str, keep_default_na=False)
assert len(raw) == EXPECTED_INPUT_ROWS, (
    f"Expected {EXPECTED_INPUT_ROWS} rows from the cleaning stage, got {len(raw)}. "
    "Its output has changed -- recheck the counts before trusting DC_TARGET."
)

# Superseded rows are resolved duplicates: the retained record already represents the site.
dc = raw[(raw["Charger_Type"] == "DC") & (raw["duplicate_status"] != "superseded")]
if not INCLUDE_FLAGGED_FOR_REVIEW:
    dc = dc[dc["duplicate_status"] != "flagged_manual_review"]
dc = dc.copy()  # own frame, so later cells can add match columns without a view warning


dc.to_csv(DC_SUBSET_CSV, index=False)
DC_TARGET = math.ceil(len(dc) / 2)

flagged_note = "included" if INCLUDE_FLAGGED_FOR_REVIEW else "excluded"
print(f"Input rows:        {len(raw)}")
print(f"DC chargers:       {len(dc)}  (flagged_manual_review {flagged_note})")
print(f"Wrote:             {DC_SUBSET_CSV}")
print(f"DC_TARGET (>=50%): {DC_TARGET}")

Input rows:        1958
DC chargers:       432  (flagged_manual_review included)
Wrote:             D:\Data Engineering\comp5339-a1\data\interim\dc_chargers.csv
DC_TARGET (>=50%): 216


## 2. Choosing an external source

The assignment allows an operator's own website, Open Charge Map, Google Maps, or another open web
API. Two facts about this dataset bear on the choice:

**Operators are concentrated.** Of the 432 DC chargers: Evie Networks 104, NRMA Electric 67,
Tesla Motors 56, Chargefox 50, JOLT 49, BP Australia 32, Ampol 32, Exploren 23 — 413 across eight
operators. Per-operator scraping means eight separate scrapers, each breaking independently when a
site changes. One aggregating API covers all eight through a single contract.

**Coverage is the binding constraint, not effort.** Whatever you choose has to reach 216 matches.
Estimate coverage on a sample of 20–30 chargers *before* committing — if the sample suggests you
will land near 30%, that is the moment to add a second source, with two weeks left rather than one.

TODO(D): record the source chosen and the reasoning. This is a required part of the report's Data
Augmentation Methodology section, alongside which attributes you are retrieving.

## 3. API contract

**You need to establish this from the live documentation.** It is not reproduced here — the
assignment requires you to "document methods and API usage", so this has to be first-hand, and API
parameters drift.

Record the following, because each one is referenced by a later cell:

- **Base endpoint** and the request method.
- **How the key is passed** — query parameter or header. Read it from the environment
  (`os.environ`); never paste a key into a cell, as this notebook is a submitted deliverable.
- **Geographic query parameters** — how to request results near a coordinate, the distance unit,
  and the result-count cap. Your matching strategy depends on being able to query by position.
- **Response fields** carrying the attributes you want (plug types, pricing, operator, bay counts),
  and which are frequently null in practice.
- **Rate limit and fair-use policy.** This sets your throttle in section 4 and belongs in the
  report. If the limit is undocumented, be conservative and say so.


## 4. Fetch, with a local cache

The assignment asks you to keep a local copy of retrieved data, since "repeated retrieval may be
time-consuming and subject to API limits". `data_acquisition.ipynb` already establishes the pattern
this notebook should follow — a `FORCE_DOWNLOAD` flag defaulting to `False`, with existing files
reused rather than refetched.

The cache is more than a convenience here: it is what lets the other three members run the pipeline
without their own API key, and what stops an exhausted quota from blocking the group.

In [4]:
# --- Open Charge Map: one bulk request, cached locally -----------------------
import time
import urllib.error
import urllib.parse
import urllib.request

OCM_ENDPOINT = "https://api.openchargemap.io/v3/poi"
OCM_COUNTRY = "AU"
USER_AGENT = "COMP5339-A1/0.1 (university assignment)"

# compact=true is deliberately NOT used: it replaces OperatorInfo and
# ConnectionType with bare IDs, losing the operator and plug-type names that are
# the whole point of the augmentation. verbose=false drops null fields but keeps
# those reference objects.
OCM_PARAMS = {
    "output": "json",
    "countrycode": OCM_COUNTRY,
    "maxresults": 10000,
    "verbose": "false",
}

# One request covers the country. Tiling by region was tested against three
# dense metro areas and returned no POIs the bulk call had missed, so per-charger
# querying would be 432 requests for no extra data -- and OCM is a non-profit.
FORCE_REFETCH = False  # True only to deliberately refresh the cache


def fetch_ocm_pois() -> list[dict]:
    """Fetch all AU POIs from Open Charge Map."""
    api_key = os.environ.get("OPENCHARGEMAP_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError(
            "OPENCHARGEMAP_API_KEY is not set. Copy .env.example to .env, add your key "
            "from openchargemap.org, and re-run the setup cell."
        )
    url = f"{OCM_ENDPOINT}?{urllib.parse.urlencode(OCM_PARAMS)}"
    request = urllib.request.Request(
        url, headers={"X-API-Key": api_key, "User-Agent": USER_AGENT}
    )
    for attempt in range(3):
        try:
            with urllib.request.urlopen(request, timeout=180) as response:
                return json.load(response)
        except urllib.error.HTTPError as error:
            if error.code == 401:
                raise RuntimeError("OCM rejected the API key (HTTP 401).") from error
            if error.code not in (429, 500, 502, 503) or attempt == 2:
                raise
            wait = 5 * (attempt + 1)  # OCM documents no rate limit; back off politely
            print(f"  HTTP {error.code}, retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError("Unreachable")


def flatten_poi(poi: dict) -> dict:
    """Reduce an OCM POI to the fields used downstream."""
    address = poi.get("AddressInfo") or {}
    connections = poi.get("Connections") or []
    plug_types = sorted({
        (c.get("ConnectionType") or {}).get("Title", "")
        for c in connections
        if (c.get("ConnectionType") or {}).get("Title")
    })
    powers = [c["PowerKW"] for c in connections if c.get("PowerKW") is not None]
    return {
        "ocm_id": poi.get("ID"),
        "ocm_uuid": poi.get("UUID"),
        "ocm_title": address.get("Title", ""),
        "ocm_latitude": address.get("Latitude"),
        "ocm_longitude": address.get("Longitude"),
        "ocm_operator": (poi.get("OperatorInfo") or {}).get("Title", ""),
        "ocm_usage_cost": poi.get("UsageCost") or "",
        "ocm_number_of_points": poi.get("NumberOfPoints"),
        "ocm_address_line1": address.get("AddressLine1", ""),
        "ocm_town": address.get("Town", ""),
        "ocm_postcode": str(address.get("Postcode") or "").strip(),
        "ocm_plug_types": "; ".join(plug_types),
        "ocm_connection_count": len(connections),
        "ocm_max_power_kw": max(powers) if powers else None,
        "ocm_date_last_verified": poi.get("DateLastVerified") or "",
    }


if EXTERNAL_CACHE.exists() and not FORCE_REFETCH:
    ocm = pd.read_csv(EXTERNAL_CACHE, keep_default_na=False)
    print(f"Using cached responses: {EXTERNAL_CACHE} ({len(ocm)} POIs)")
else:
    print(f"Fetching from {OCM_ENDPOINT} ...")
    raw_pois = fetch_ocm_pois()
    ocm = pd.DataFrame([flatten_poi(p) for p in raw_pois])
    ocm = ocm.dropna(subset=["ocm_latitude", "ocm_longitude"])
    ocm.to_csv(EXTERNAL_CACHE, index=False, encoding="utf-8", lineterminator="\n")
    print(f"Fetched {len(raw_pois)} POIs, cached {len(ocm)} with coordinates -> {EXTERNAL_CACHE}")

print(f"POIs available for matching: {len(ocm)}")

Using cached responses: D:\Data Engineering\comp5339-a1\data\external\external_charger_details.csv (1366 POIs)
POIs available for matching: 1366


## 4b. Second source: OpenStreetMap via the Overpass API

Open Charge Map alone matched 190 of 432 DC chargers (44%), short of the 216 target. The 242
unmatched are not in OCM's Australian dataset at any distance -- verified by paginating the full
dataset, by the nearest-POI distance distribution (median 1.8 km), and by checking for a shared
identifier (1,365 of 1,366 OCM records are community-contributed with no external reference).

**OpenStreetMap is used as the second source.** A single Overpass query for
`amenity=charging_station` across the NSW bounding box returns ~650 stations and adds **140**
matches, taking the total to **330 / 432 (76%)**. It covers precisely what OCM missed -- JOLT (where
OCM's hit rate was 10%), Evie, Ampol -- so the two sources are complementary rather than redundant.

It was chosen over scraping an operator's website because it needs no key, no registration and no
scraping: Overpass is a documented bulk API, and the assignment permits "another open Web API".

**Both sources are kept.** OSM's `fee` tag is a yes/no flag present on 29% of added rows, whereas
OCM's `UsageCost` carries real tariffs (`$0.55/kWh`) on 81% of its matches. OCM remains the pricing
source, which matters for Assignment 2's price monitoring.

*Attribution: OpenStreetMap contributors, ODbL.*

In [5]:
# --- OpenStreetMap charging stations via Overpass ----------------------------
NSW_BBOX = (-37.6, 140.9, -28.1, 153.7)  # south, west, north, east
OVERPASS_TILES = 3  # NSW is split into a 3x3 grid; see note below

# Overpass offers no SLA. A single query over the whole state times out under
# load (observed: HTTP 504 from the canonical instance, then a read timeout from
# the mirror). Splitting into tiles makes each request cheap enough to succeed,
# and a slow tile costs one retry rather than the whole fetch.
OVERPASS_MIRRORS = (
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
)
OVERPASS_TIMEOUT_S = 120


def overpass_tiles(bbox, n):
    """Split a bounding box into an n x n grid of (south, west, north, east)."""
    south, west, north, east = bbox
    lat_step = (north - south) / n
    lon_step = (east - west) / n
    return [
        (south + r * lat_step, west + c * lon_step,
         south + (r + 1) * lat_step, west + (c + 1) * lon_step)
        for r in range(n) for c in range(n)
    ]


def overpass_request(query: str) -> list[dict]:
    """POST a query, trying each mirror in turn. Raises if all fail."""
    payload = query.encode("utf-8")
    failures = []
    for mirror in OVERPASS_MIRRORS:
        try:
            request = urllib.request.Request(
                mirror, data=payload, headers={"User-Agent": USER_AGENT}
            )
            with urllib.request.urlopen(request, timeout=OVERPASS_TIMEOUT_S) as response:
                return json.load(response)["elements"]
        except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError, OSError) as error:
            failures.append(f"{mirror.split('/')[2]}: {error}")
    raise RuntimeError("All Overpass mirrors failed:\n    " + "\n    ".join(failures))


def fetch_osm_stations() -> list[dict]:
    """Fetch NSW charging stations tile by tile, de-duplicated by OSM node id."""
    elements = {}
    tiles = overpass_tiles(NSW_BBOX, OVERPASS_TILES)
    for index, (south, west, north, east) in enumerate(tiles, start=1):
        query = (
            f'[out:json][timeout:90];'
            f'node["amenity"="charging_station"]({south},{west},{north},{east});'
            f'out body;'
        )
        found = overpass_request(query)
        elements.update({e["id"]: e for e in found})
        print(f"  tile {index}/{len(tiles)}: {len(found):>4} stations  (total {len(elements)})")
        time.sleep(1)  # Overpass is volunteer-run; do not hammer it
    return list(elements.values())


def flatten_osm(element: dict) -> dict:
    """Reduce an Overpass node to the fields used downstream."""
    tags = element.get("tags") or {}
    sockets = sorted(
        tag.split(":", 1)[1] for tag in tags if tag.startswith("socket:") and tags[tag] != "no"
    )
    return {
        "osm_id": element.get("id"),
        "osm_latitude": element.get("lat"),
        "osm_longitude": element.get("lon"),
        "osm_name": tags.get("name", ""),
        "osm_operator": tags.get("operator") or tags.get("brand") or "",
        "osm_capacity": tags.get("capacity", ""),
        "osm_fee": tags.get("fee", ""),
        "osm_sockets": "; ".join(sockets),
    }


if OSM_CACHE.exists() and not FORCE_REFETCH:
    osm = pd.read_csv(OSM_CACHE, keep_default_na=False)
    print(f"Using cached OSM responses: {OSM_CACHE} ({len(osm)} stations)")
else:
    print("Fetching from Overpass ...")
    elements = fetch_osm_stations()
    osm = pd.DataFrame([flatten_osm(e) for e in elements])
    osm = osm.dropna(subset=["osm_latitude", "osm_longitude"])
    osm.to_csv(OSM_CACHE, index=False, encoding="utf-8", lineterminator="\n")
    print(f"Fetched {len(elements)} nodes, cached {len(osm)} with coordinates -> {OSM_CACHE}")

print(f"OSM stations available for matching: {len(osm)}")

Using cached OSM responses: D:\Data Engineering\comp5339-a1\data\external\osm_charging_stations.csv (653 stations)
OSM stations available for matching: 653


## 5. Matching external records to TfNSW chargers

The two datasets share no identifier, so the link must be inferred. **The data forces the strategy
here:**

| field | populated on DC rows |
|---|---|
| `Latitude` / `Longitude` | **432 / 432** |
| `Station_address` | 432 / 432, but inconsistent in form |
| `Station_name` | **1 / 432** |

So **name matching is not available** — a fact worth stating plainly in the report, since the
assignment asks which strategy you used and name matching is the obvious first guess. Addresses vary
in structure (`"01 Wallgrove Road, Sydney, 2766"` — no state, and the suburb is wrong — alongside
`"1 - 7 Ross St, Wilcannia NSW 2836, Australia"`), which makes them a weak tiebreaker at best.

**Coordinate proximity is your primary key.** Two failure modes to design against:

- **Multiple chargers at one site.** A service centre with six bays may appear as several TfNSW rows
  within metres of each other. Any radius large enough to absorb coordinate error will capture all
  of them, so decide what a match *means*: nearest single record, or all records within the radius?
- **False precision.** Source coordinates carry eight decimal places (`-33.81100405`) — roughly
  millimetre resolution, which the underlying data almost certainly does not have. Do not let the
  formatting talk you into a tight radius.

In [6]:
# --- Match external records to TfNSW DC chargers -----------------------------
#
# Station_name is populated on 1 of 432 DC rows and addresses are inconsistent in
# form, so coordinate proximity is the primary key. Three rules, applied in order,
# so each charger is attributed to the first that succeeds:
#
#   1. ocm / proximity  -- nearest OCM POI within MATCH_RADIUS_M
#   2. ocm / address    -- same street NUMBER and NAME within ADDRESS_MATCH_MAX_M,
#                          recovering sites whose coordinates disagree between the
#                          datasets (e.g. 173 The Vineyards Rd, matched at 1284 m)
#   3. osm / proximity  -- nearest OSM station within MATCH_RADIUS_M
#
# OCM is tried first so its result stays reproducible and OSM is purely additive.
#
# REJECTED: a looser address rule -- any shared street token within 2 km -- reached
# 260 matches and would have cleared the target, but inspection showed it was mostly
# false. Town names leak into street tokens ("2 Stewart St, Lithgow" matched "Club
# Lithgow | 2c Lithgow St") and highway names carry almost no location information
# ("66 Princes Hwy, Wollongong" matched "224 Princes Hwy, Fairy Meadow"). Requiring
# the street number as well removes those.
MATCH_RADIUS_M = 200
ADDRESS_MATCH_MAX_M = 5_000

EARTH_RADIUS_M = 6_371_000.0

STREET_SUFFIXES = {
    "rd": "road", "st": "street", "ave": "avenue", "av": "avenue", "hwy": "highway",
    "dr": "drive", "pde": "parade", "cres": "crescent", "ct": "court", "pl": "place",
    "tce": "terrace", "ln": "lane", "blvd": "boulevard", "bvd": "boulevard", "wy": "way",
}


def parse_street(address: str) -> tuple[str, str]:
    """'173 The Vineyards Rd, Lake George' -> ('173', 'the vineyards road').

    Only the text before the first comma is used; everything after is suburb and
    postcode, which is what made looser token matching produce false pairs.
    """
    head = str(address).split(",")[0].strip()
    match = re.match(r"\s*(\d+)\s*[-/]?\s*\d*\s+(.*)", head)
    if not match:
        return ("", "")
    number = match.group(1).lstrip("0") or "0"
    words = [STREET_SUFFIXES.get(w, w) for w in re.findall(r"[a-z]+", match.group(2).lower())]
    return (number, " ".join(words))


def haversine_metres(lat1, lon1, lat2, lon2):
    """Great-circle distance in metres. lat2/lon2 may be arrays."""
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    d_phi = np.radians(lat2 - lat1)
    d_lambda = np.radians(lon2 - lon1)
    a = np.sin(d_phi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(d_lambda / 2) ** 2
    return 2 * EARTH_RADIUS_M * np.arcsin(np.sqrt(a))


ocm_lat = ocm["ocm_latitude"].to_numpy(dtype=float)
ocm_lon = ocm["ocm_longitude"].to_numpy(dtype=float)
ocm_parsed = [parse_street(a) for a in ocm["ocm_address_line1"]]
ocm_number = np.array([s[0] for s in ocm_parsed])
ocm_name = np.array([s[1] for s in ocm_parsed])

osm_lat = osm["osm_latitude"].to_numpy(dtype=float)
osm_lon = osm["osm_longitude"].to_numpy(dtype=float)

records = []
for row in dc.itertuples(index=False):
    lat, lon = float(row.Latitude), float(row.Longitude)
    ocm_distances = haversine_metres(lat, lon, ocm_lat, ocm_lon)
    osm_distances = haversine_metres(lat, lon, osm_lat, osm_lon)
    ocm_nearest = int(np.argmin(ocm_distances))
    osm_nearest = int(np.argmin(osm_distances))
    within_radius = int((ocm_distances <= MATCH_RADIUS_M).sum())

    source = method = None
    pick = distance = None

    if ocm_distances[ocm_nearest] <= MATCH_RADIUS_M:
        source, method, pick = "ocm", "proximity", ocm_nearest
        distance = ocm_distances[ocm_nearest]
    else:
        number, name = parse_street(row.Station_address)
        corroborated = np.where(
            (ocm_number == number) & (ocm_name == name)
            & (number != "") & (name != "")
            & (ocm_distances <= ADDRESS_MATCH_MAX_M)
        )[0]
        if len(corroborated):
            pick = int(corroborated[np.argmin(ocm_distances[corroborated])])
            source, method, distance = "ocm", "address", ocm_distances[pick]
        elif osm_distances[osm_nearest] <= MATCH_RADIUS_M:
            source, method, pick = "osm", "proximity", osm_nearest
            distance = osm_distances[osm_nearest]

    record = {
        "charger_record_id": row.charger_record_id,
        "match_source": source or "none",
        "match_method": method or "unmatched",
        "matched": source is not None,
        "match_distance_m": round(float(distance), 1) if source else None,
        "nearest_ocm_distance_m": round(float(ocm_distances[ocm_nearest]), 1),
        "nearest_osm_distance_m": round(float(osm_distances[osm_nearest]), 1),
        "match_candidates_within_radius": within_radius,
        "match_is_ambiguous": within_radius > 1,
    }
    if source == "ocm":
        record.update(ocm.iloc[pick].to_dict())
    elif source == "osm":
        record.update(osm.iloc[pick].to_dict())
    records.append(record)

matches = pd.DataFrame(records)

# Source-neutral columns, so the storage stage does not need to know which API a
# value came from. match_source retains the provenance.
matches["aug_operator"] = matches.get("ocm_operator", "").fillna("").replace("", pd.NA) \
    .fillna(matches.get("osm_operator", "")).fillna("")
matches["aug_plug_types"] = matches.get("ocm_plug_types", "").fillna("").replace("", pd.NA) \
    .fillna(matches.get("osm_sockets", "")).fillna("")
matches["aug_bays"] = matches.get("ocm_number_of_points", pd.NA) \
    .fillna(pd.to_numeric(matches.get("osm_capacity", pd.NA), errors="coerce"))
matches["aug_cost"] = matches.get("ocm_usage_cost", "").fillna("").replace("", pd.NA) \
    .fillna(matches.get("osm_fee", "").map({"yes": "fee (OSM, unpriced)", "no": "free (OSM)"})) \
    .fillna("")

augmented = dc.merge(matches, on="charger_record_id", how="left", validate="one_to_one")
assert len(augmented) == len(dc), "merge changed the row count"
augmented.to_csv(AUGMENTED_CSV, index=False, encoding="utf-8", lineterminator="\n")

matched_count = int(augmented["matched"].sum())
by_source = augmented.loc[augmented["matched"], "match_source"].value_counts()
print(f"Matched: {matched_count} / {len(dc)} ({matched_count / len(dc):.1%})")
print(f"    from OCM  {int(by_source.get('ocm', 0))}"
      f"  (proximity {int((augmented['match_method'] == 'proximity').sum() - by_source.get('osm', 0))}"
      f", address {int((augmented['match_method'] == 'address').sum())})")
print(f"    from OSM  {int(by_source.get('osm', 0))}")
print(f"Target (>=50%): {DC_TARGET}  -> "
      f"{'MET' if matched_count >= DC_TARGET else f'SHORT by {DC_TARGET - matched_count}'}")
print(f"Wrote: {AUGMENTED_CSV}")

Matched: 330 / 432 (76.4%)
    from OCM  190  (proximity 184, address 6)
    from OSM  140
Target (>=50%): 216  -> MET
Wrote: D:\Data Engineering\comp5339-a1\data\interim\dc_chargers_augmented.csv


## 6. Match rate and quality

`integration_quality_B.ipynb` writes `cleaning_quality_summary.json` so the report quotes a
regenerable artefact instead of numbers retyped from cell output. Follow that pattern — B also needs
these figures for the quality section covering the *augmented* dataset.

Report at minimum:

- **Match rate against the 432 denominator**, and whether it clears **216**. This is the graded
  number.
- **Per-attribute fill rate.** A match that returns a row of nulls is not augmentation; the
  assignment asks for attributes actually added.
- **Distance distribution** of accepted matches — a long tail near your threshold means the
  threshold is doing more work than you think.
- **Unmatched and ambiguous counts**, and any pattern in them (one operator? rural sites?). A
  systematic gap is a finding worth reporting, not an embarrassment.

In [7]:
# --- Match rate and attribute quality ----------------------------------------
matched_rows = augmented[augmented["matched"].fillna(False)]

# Sensitivity of the proximity rule to its radius, per source -- justifies
# MATCH_RADIUS_M rather than asserting it.
radius_sensitivity = {
    f"{r}m": {
        "ocm": int((augmented["nearest_ocm_distance_m"] <= r).sum()),
        "osm": int((augmented["nearest_osm_distance_m"] <= r).sum()),
        "either": int(((augmented["nearest_ocm_distance_m"] <= r)
                       | (augmented["nearest_osm_distance_m"] <= r)).sum()),
    }
    for r in (50, 100, 150, 200, 300, 500, 1000)
}

SOURCE_NEUTRAL_ATTRIBUTES = ["aug_operator", "aug_plug_types", "aug_bays", "aug_cost"]
attribute_fill = {
    column: int(matched_rows[column].replace("", pd.NA).notna().sum())
    for column in SOURCE_NEUTRAL_ATTRIBUTES
}

summary = {
    "dc_chargers": len(augmented),
    "target_50_percent": DC_TARGET,
    "matched": len(matched_rows),
    "match_rate": round(len(matched_rows) / len(augmented), 4),
    "target_met": bool(len(matched_rows) >= DC_TARGET),
    "matched_by_source": matched_rows["match_source"].value_counts().to_dict(),
    "matched_by_method": matched_rows["match_method"].value_counts().to_dict(),
    "match_radius_m": MATCH_RADIUS_M,
    "address_match_max_m": ADDRESS_MATCH_MAX_M,
    "ambiguous_matches": int(augmented["match_is_ambiguous"].fillna(False).sum()),
    "radius_sensitivity": radius_sensitivity,
    "attribute_fill_among_matched": attribute_fill,
    "unmatched_by_operator": (
        augmented.loc[~augmented["matched"].fillna(False), "Operator"]
        .value_counts().to_dict()
    ),
    "sources": {
        "ocm": {"records": len(ocm), "endpoint": OCM_ENDPOINT},
        "osm": {"records": len(osm), "endpoint": "Overpass API (OpenStreetMap, ODbL)"},
    },
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Match rate: {summary['match_rate']:.1%} "
      f"({summary['matched']}/{summary['dc_chargers']}, target {DC_TARGET}) "
      f"-> {'MET' if summary['target_met'] else 'NOT MET'}\n")
print("By source:", summary["matched_by_source"])
print("\nRadius sensitivity (nearest candidate, by source):")
print(f"  {'radius':>7}{'OCM':>7}{'OSM':>7}{'either':>9}")
for radius, counts in radius_sensitivity.items():
    print(f"  {radius:>7}{counts['ocm']:>7}{counts['osm']:>7}{counts['either']:>9}")
print("\nAttribute fill among matched:")
for column, count in attribute_fill.items():
    print(f"  {column:<18}{count:>4} / {len(matched_rows)}  "
          f"({count / max(len(matched_rows), 1):.0%})")
print("\nEvery address-corroborated match (few enough to check by hand):")
for r in augmented[augmented["match_method"] == "address"].itertuples():
    print(f"  {r.match_distance_m:>7.0f} m  {r.Station_address[:42]:<44} -> {str(r.ocm_title)[:32]}")
print("\nLargest remaining unmatched operators:")
for operator, count in list(summary["unmatched_by_operator"].items())[:5]:
    print(f"  {operator:<24}{count}")
print(f"\nWrote: {SUMMARY_JSON}")

Match rate: 76.4% (330/432, target 216) -> MET

By source: {'ocm': 190, 'osm': 140}

Radius sensitivity (nearest candidate, by source):
   radius    OCM    OSM   either
      50m    123    223      247
     100m    163    269      295
     150m    174    283      310
     200m    184    297      326
     300m    201    305      336
     500m    214    327      353
    1000m    240    344      368

Attribute fill among matched:
  aug_operator       326 / 330  (99%)
  aug_plug_types     278 / 330  (84%)
  aug_bays           289 / 330  (88%)
  aug_cost           195 / 330  (59%)

Every address-corroborated match (few enough to check by hand):
      225 m  128 Flood Street, Leichhardt NSW 2040        -> Leichhardt Marketplace
      723 m  152 Pacific Hwy, 36 Josephson St, Swansea    -> Swansea
      237 m  17 Stanley St, Sydney, 2210                  -> Stanley St Peakhurst NSW
     1284 m  173 The Vineyards Rd, Lake George, 2581      -> Lake George Winery
      325 m  211 Lake Entrance Rd

## Hand-off

1. `data/interim/dc_chargers.csv` — the DC subset, and the shared denominator for the ≥50% target.
2. `data/external/external_charger_details.csv` — cached responses. **Commit this**; it is the one
   data directory that is tracked, and it lets the group run without a key.
3. `data/interim/dc_chargers_augmented.csv` — chargers with external attributes attached, consumed
   by the storage stage.
4. `data/interim/augmentation_summary.json` — figures for B's quality section.

Once the logic here is settled, promote the reusable parts into `src/augmentation/fetch.py` and
`src/augmentation/matching.py`, and implement `src/augmentation/run()` — the same notebook-first,
then-promote order the other two notebooks follow. The pipeline entry point in `main.py` calls that
`run()`, not this notebook.

Then tell C and D-storage what changed: the augmented attributes need somewhere to live in
`sql/schema.sql`.